# Near-miss financial abstention: full pipeline

Code: https://github.com/phwithon/NLP_HW2

1. Setup (get the code)
2. Reproduce every table and figure from the included predictions (**no GPU needed**, about 1 minute)
3. Pilot (GPU; separate items, used only to check the prompt)
4. Main experiment (GPU; regenerates the predictions)
5. Look at the outputs and download them

For steps 3-4, select Runtime → Change runtime type → **T4 GPU** before running.

## 1. Setup
Run **one** of the two cells below. Option A clones the public repository; option B uploads the repository as a zip file (e.g. GitHub → Code → Download ZIP).

In [ ]:
# Option A: clone from GitHub
%cd /content
!rm -rf /content/NLP_HW2
!git clone -q https://github.com/phwithon/NLP_HW2
%cd /content/NLP_HW2
!pip install -q -r requirements.txt
!ls

In [ ]:
# Option B: upload the repository as a zip file
import glob, os, shutil, zipfile
from google.colab import files

%cd /content
uploaded = files.upload()  # choose the zip
zip_name = next(n for n in uploaded if n.endswith('.zip'))
shutil.rmtree('/content/near-miss-fin', ignore_errors=True)
zipfile.ZipFile(zip_name).extractall('/content/near-miss-fin')

# work in the folder that holds config.yaml, wherever it sits inside the zip
root = os.path.dirname(sorted(glob.glob('/content/near-miss-fin/**/config.yaml', recursive=True), key=len)[0])
%cd $root
!pip install -q -r requirements.txt
!ls

## 2. Reproduce all tables and the figure (no GPU)
Uses the predictions included in `results/`, `results_pilot/` and `results_overlap/`. The benchmark files are regenerated deterministically and should be identical to the included ones.

In [ ]:
!python src/generate.py --split main
!python src/generate.py --split pilot
!python src/generate.py --split main --templates overlap --out data/items_overlap.jsonl
!git diff --stat -- data 2>/dev/null || true   # empty output = data identical (option A)
!python src/analyze.py
!python src/extra.py
!python src/analyze.py --results results_pilot --data data/pilot.jsonl --out outputs_pilot

## 3. Pilot (GPU)
20 separate items (seed 7). Criteria fixed in the proposal:
- ANS accuracy should be at least 80% (otherwise failures reflect reading, not abstention)
- the presence baseline (IRR) should be neither at floor nor at ceiling

In our runs ANS accuracy was 100%, but IRR was already at 0% under both prompts, so the second criterion was not met; the paper reports this and keeps both prompts. Near-miss substitution ranged from 0% to 100% across conditions.

In [ ]:
!python src/generate.py --split pilot
!python src/run.py --data data/pilot.jsonl --all
!python src/analyze.py --results results_pilot --data data/pilot.jsonl --out outputs_pilot

In [ ]:
import pandas as pd
p = pd.read_csv('outputs_pilot/summary.csv')
p.pivot_table(index='condition', columns=['model', 'prompt'], values='substitution').round(2)

## 4. Main experiment (GPU)
120 items × 11 conditions × 2 models × 2 prompts = 5,280 prompts (about 6 minutes of inference on a T4, plus model download). This overwrites the included `results/`; scores can differ in the last digits across GPUs and library versions (see README).

In [ ]:
!python src/generate.py --split main
!python src/run.py --data data/items.jsonl --all
!python src/analyze.py
!python src/extra.py   # style table, overlap check (uses the included results_overlap/), error composition

Optional: regenerate the predictions for the overlap robustness check (another 5,280 prompts).

In [ ]:
# !python src/generate.py --split main --templates overlap --out data/items_overlap.jsonl
# !python src/run.py --data data/items_overlap.jsonl --all --results results_overlap
# !python src/extra.py

## 5. Outputs

In [ ]:
print(open('outputs/hypotheses.md').read())
print(open('outputs/table_irr.tex').read())
from IPython.display import Image, display
display(Image('outputs/fig_delta.png', width=500))

In [ ]:
import pandas as pd
s = pd.read_csv('outputs/summary.csv')
s.pivot_table(index='condition', columns=['model', 'prompt'], values=['substitution', 'abstention']).round(3)

In [ ]:
!cat results/*.meta.json
!zip -qr near_miss_outputs.zip data results results_pilot results_overlap outputs outputs_pilot
from google.colab import files
files.download('near_miss_outputs.zip')